# SwiftServe real GPU replica node (Colab or Kaggle, free T4)

This notebook turns this free-tier GPU session into **one real replica node**
in a genuinely multi-node SwiftServe deployment: it runs a real vLLM server
(real Qwen weights, real GPU inference -- not a stand-in), fronts it with
SwiftServe's chaos-capable sidecar (`swiftserve/replica_sidecar.py`), and
opens a public tunnel so the SwiftServe router -- running wherever you
control it, e.g. your own laptop -- can reach it as one entry in
`SWIFTSERVE_REPLICAS`.

**Run this same notebook once per GPU node you want** (a second Colab
account, a Kaggle account, etc. -- each free-tier account gives you one
T4). Two or three separate real accounts running this notebook is what
makes the cluster genuinely multi-node: separate physical machines,
separate network paths, connected over the public internet rather than
`localhost`.

Carries forward two real bugs found and fixed while first building this
project on a single Colab GPU:
1. A CUDA/torchaudio version mismatch crashed vLLM instantly, but the
   naive health-check loop kept retrying and looked like a hang instead of
   a crash -- fixed here by uninstalling torchaudio and by having the wait
   loop check the process's exit status, not just its HTTP health.
2. Launching multiple vLLM processes on one GPU at once caused a
   memory-contention crash -- irrelevant to this notebook specifically
   (each node runs exactly one vLLM process), but the same
   `--gpu-memory-utilization` caution applies if you adapt this to share a
   GPU across replicas again.
**Kaggle-specific setup** (Colab needs none of this): in the notebook's
Settings panel, turn on **Internet** and set **Accelerator** to a GPU
(T4 x2 or P100) before running any cell below -- Kaggle notebooks default
to no internet access, which breaks the git clone and pip install steps.


## 1. Configure this node

Edit `REPLICA_TAG` and `ADMIN_TOKEN` per node -- everything else can stay default.

In [ ]:
import secrets

MODEL = "Qwen/Qwen2.5-3B-Instruct"   # real prefill cost on a free T4; drop to 1.5B if VRAM is tight
REPLICA_TAG = "replica-a"            # CHANGE PER ACCOUNT: replica-a / -b / -c / -d
BRANCH = "claude/friendly-turing-v9y7vj"   # branch carrying the block-level prefix index

# THE flag that decides whether the demo shows anything. A T4 at 0.90
# utilization holds ~200k tokens of KV cache (~100 distinct 2000-token
# system prompts) while the demo workload has 6 -- so by default nothing
# ever evicts, every policy gets a cache hit, and all policies tie.
# 375 blocks = 6000 tokens = 0.5x the 6-app x 2000-token working set.
# See DEMO_RUNBOOK.md section 1 for other workload sizes.
KV_BLOCKS = 375

VLLM_PORT = 8000
SIDECAR_PORT = 9000
# Leave blank to auto-generate a random token (printed below) -- whatever it
# is, you'll need it on the router side to hit this node's /chaos/* endpoints.
ADMIN_TOKEN = "" or secrets.token_hex(8)
print(f"REPLICA_TAG={REPLICA_TAG}  MODEL={MODEL}  KV_BLOCKS={KV_BLOCKS}")
print(f"ADMIN_TOKEN={ADMIN_TOKEN}  <- copy this, the chaos scenario runner needs it")


## 2. Get the SwiftServe code onto this machine

In [ ]:
# --branch matters: the block-level prefix index and the prefix_aware
# policy are not on main yet.
!git clone --depth 1 --branch {BRANCH} https://github.com/Akshataram/distrouter.git
%cd distrouter
!git log --oneline -1


## 3. Install dependencies

Installs vLLM plus the router/sidecar's own (CPU-only) dependencies, then
removes `torchaudio` -- on Colab/Kaggle's preinstalled CUDA images its
pinned version is frequently mismatched against the CUDA build vLLM wants,
and vLLM does not need it. Without this step, vLLM can crash on startup
with a CUDA symbol error that looks nothing like "wrong torchaudio".

In [ ]:
!pip install -q vllm
!pip install -q -r requirements.txt
!pip uninstall -y -q torchaudio || true


## 4. Launch vLLM + the chaos-capable sidecar

The sidecar is started with `--supervise`, so it directly owns the vLLM
process as a child it can `SIGKILL` and relaunch on command -- this is what
lets a chaos scenario (`scripts/chaos_runner.py`, run from wherever you
drive the demo) actually kill and recover *this specific real GPU node*
over the network, not just simulate it.

The wait loop below polls the sidecar's own `/health` (which proxies
through to real vLLM) and treats the supervised process dying as an
immediate hard failure instead of retrying forever -- if this cell hangs
past a minute or two on stock Colab hardware, the printed log tail is
almost always a CUDA/OOM error, not a slow load.

In [ ]:
import subprocess, time, os, textwrap, urllib.request, urllib.error

LOG_DIR = "logs"
os.makedirs(LOG_DIR, exist_ok=True)
sidecar_log = open(f"{LOG_DIR}/sidecar.log", "w")

supervise_cmd = (
    f"python -m vllm.entrypoints.openai.api_server "
    f"--model {MODEL} --port {VLLM_PORT} --enable-prefix-caching "
    f"--enable-prompt-tokens-details --block-size 16 --max-num-seqs 32 "
    f"--max-model-len 4096 --dtype half --gpu-memory-utilization 0.90 "
    f"--num-gpu-blocks-override {KV_BLOCKS}"
)

sidecar_proc = subprocess.Popen(
    [
        "python", "-m", "swiftserve.replica_sidecar",
        "--upstream-url", f"http://127.0.0.1:{VLLM_PORT}",
        "--port", str(SIDECAR_PORT),
        "--admin-token", ADMIN_TOKEN,
        "--supervise", supervise_cmd,
    ],
    stdout=sidecar_log, stderr=subprocess.STDOUT,
)
print(f"sidecar pid={sidecar_proc.pid}, supervising: {supervise_cmd}")

def tail_log(path, n=40):
    with open(path) as f:
        lines = f.readlines()
    return "".join(lines[-n:])

deadline = time.monotonic() + 600  # real Qwen weight download + load can take a few minutes on a cold node
healthy = False
while time.monotonic() < deadline:
    if sidecar_proc.poll() is not None:
        print("!! sidecar process died -- log tail:\n" + tail_log(f"{LOG_DIR}/sidecar.log"))
        break
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{SIDECAR_PORT}/health", timeout=3) as resp:
            if resp.status == 200:
                healthy = True
                break
    except (urllib.error.URLError, ConnectionError, TimeoutError):
        pass
    time.sleep(5)

if healthy:
    print(f"replica '{REPLICA_TAG}' is up: real vLLM serving {MODEL}, fronted by the sidecar on port {SIDECAR_PORT}")
else:
    print("!! never became healthy within the timeout -- log tail:\n" + tail_log(f"{LOG_DIR}/sidecar.log"))


## 5. Sanity-check real inference through the sidecar

Confirms the whole local chain works (sidecar -> real vLLM -> real Qwen
generation) before exposing it publicly.

In [ ]:
import json, urllib.request

req = urllib.request.Request(
    f"http://127.0.0.1:{SIDECAR_PORT}/v1/chat/completions",
    data=json.dumps({"model": MODEL, "messages": [{"role": "user", "content": "Say hello in five words."}], "max_tokens": 32}).encode(),
    headers={"Content-Type": "application/json"},
    method="POST",
)
with urllib.request.urlopen(req, timeout=60) as resp:
    body = json.loads(resp.read())
print(body["choices"][0]["message"]["content"])


## 6. Prove prefix caching works ON THIS NODE

Two checks, before the router is involved at all. If either fails, fix it
here -- debugging this through the router later is much harder.

1. **Did `--num-gpu-blocks-override` take?** vLLM logs its real block
   count at startup. If it does not match `KV_BLOCKS`, the flag was
   ignored and there will be no cache pressure, so every routing policy
   will tie.
2. **Does a repeated prefix actually get reused?** Send the same
   2000-token system prompt twice and compare TTFT and the engine's own
   `cached_tokens`. The second call should be dramatically faster with
   `cached_tokens` > 0. If `cached_tokens` is missing or 0, then
   `--enable-prompt-tokens-details` is not in effect and the whole
   measurement story is broken.


In [ ]:
import json, time, urllib.request, re

# --- check 1: the real KV block count ------------------------------------
log = tail_log(f"{LOG_DIR}/sidecar.log", 400)
m = re.findall(r"GPU (?:KV cache size|blocks)[^0-9]*([0-9,]+)", log)
print(f"KV_BLOCKS requested : {KV_BLOCKS}")
print(f"vLLM reported       : {m if m else 'NOT FOUND in log -- grep manually'}")
print("   (if these disagree, --num-gpu-blocks-override did not take effect)\n")

# --- check 2: cold vs warm on the same prefix ----------------------------
BIG_SYSTEM = "You are a meticulous customer support agent. " * 200  # ~2000 tokens

def call(user, max_tokens=16):
    body = json.dumps({
        "model": MODEL,
        "messages": [{"role": "system", "content": BIG_SYSTEM},
                     {"role": "user", "content": user}],
        "max_tokens": max_tokens,
        "stream": True,
        "stream_options": {"include_usage": True},
    }).encode()
    req = urllib.request.Request(
        f"http://127.0.0.1:{SIDECAR_PORT}/v1/chat/completions", data=body,
        headers={"Content-Type": "application/json"}, method="POST")
    t0 = time.monotonic(); ttft = None; usage = None
    with urllib.request.urlopen(req, timeout=180) as resp:
        for raw in resp:
            line = raw.decode("utf-8", "replace").strip()
            if not line.startswith("data:"):
                continue
            payload = line[5:].strip()
            if payload == "[DONE]":
                break
            chunk = json.loads(payload)
            choices = chunk.get("choices") or []
            if choices and (choices[0].get("delta") or {}).get("content") and ttft is None:
                ttft = (time.monotonic() - t0) * 1000
            if chunk.get("usage"):
                usage = chunk["usage"]
    return ttft, usage

print(f"{'call':<28}{'TTFT':>10}{'prompt_tok':>12}{'cached_tok':>12}")
rows = []
for label, q in [("1st (cold prefix)", "reset my password"),
                 ("2nd (same system prompt)", "where is my invoice"),
                 ("3rd (same system prompt)", "cancel my plan")]:
    ttft, usage = call(q)
    cached = (usage or {}).get("prompt_tokens_details", {}).get("cached_tokens")
    prompt = (usage or {}).get("prompt_tokens")
    rows.append((label, ttft, prompt, cached))
    print(f"{label:<28}{ttft:>8.0f}ms{str(prompt):>12}{str(cached):>12}")

warm = [r for r in rows[1:] if r[3]]
if not warm:
    print("\n!! cached_tokens is 0/None on repeat calls.")
    print("   Either --enable-prompt-tokens-details or --enable-prefix-caching")
    print("   is not in effect. Fix this before running the demo.")
elif rows[0][1] and warm[0][1] and rows[0][1] / warm[0][1] > 1.5:
    print(f"\nOK: prefix caching is live on this node "
          f"({rows[0][1] / warm[0][1]:.1f}x lower TTFT on the warm call).")
else:
    print("\n?? cached_tokens looks right but TTFT barely moved -- the prompt may")
    print("   be too short for prefill to matter. Try a longer BIG_SYSTEM.")


## 7. Open a public tunnel to this node

Uses a Cloudflare **quick tunnel**: no account or signup needed, but the
URL is random and only lives as long as this cell's process runs -- if you
need a stable URL across restarts, that requires a free Cloudflare account
and a named tunnel instead (see
https://developers.cloudflare.com/cloudflare-one/connections/connect-networks/do-more-with-tunnels/named-tunnels/).

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

import subprocess, re, time

tunnel_log = open(f"{LOG_DIR}/cloudflared.log", "w")
tunnel_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", f"http://127.0.0.1:{SIDECAR_PORT}"],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)

public_url = None
deadline = time.monotonic() + 60
while time.monotonic() < deadline:
    with open(f"{LOG_DIR}/cloudflared.log") as f:
        text = f.read()
    match = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", text)
    if match:
        public_url = match.group(0)
        break
    time.sleep(2)

if public_url:
    print(f"replica '{REPLICA_TAG}' is reachable at: {public_url}")
    print(f"admin token: {ADMIN_TOKEN}")
    print()
    print("On the machine running the SwiftServe router, add this URL as one")
    print("entry in SWIFTSERVE_REPLICAS (comma-separated with the other nodes'")
    print("URLs), and use the admin token above for scripts/chaos_runner.py's")
    print("--admin-token when targeting this replica.")
else:
    print("!! tunnel URL never appeared -- log tail:\n" + tail_log(f"{LOG_DIR}/cloudflared.log"))


## 8. Keep this cell running

Colab/Kaggle reclaim the session once it goes idle or the notebook stops
executing -- run the cell below (or just leave the notebook open and
periodically re-run a lightweight cell) to keep this node's tunnel alive
for the duration of your demo. Stopping this cell (or the session timing
out) takes this replica off the router's `SWIFTSERVE_REPLICAS` list as far
as reachability goes -- SwiftServe's circuit breaker will correctly detect
that and route around it, which is itself worth demonstrating.

In [ ]:
import time
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("stopped by user")
